# Intent Classifier v2 — Fixed

**Fixes applied over the original notebook:**

| Issue | Fix |
|---|---|
| Wrong labels at inference | Save `id2label` JSON and reload it; use `int(k)` when reloading |
| No softmax → wrong probabilities | Add `torch.softmax(logits, dim=-1)` in inference |
| No padding/truncation parity | Use identical `MAX_LENGTH=64`, `padding='max_length'` in inference |
| Overfitting (30 epochs) | Reduce to 10 epochs + `label_smoothing_factor=0.1` |
| No held-out test set | 3-way 80/10/10 split |
| No smoke test | `predict_intent()` helper with real phrases |

## Step 1 — Install Dependencies

In [ ]:
!pip install -q transformers datasets evaluate scikit-learn accelerate

## Step 2 — Imports & Configuration

In [ ]:
import os, json, random
import numpy as np
import pandas as pd
import torch

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
import evaluate

MODEL_NAME = 'distilbert-base-multilingual-cased'
MAX_LENGTH = 64   # MUST be consistent between training and inference

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print('✅ Imports done')
print(f'GPU available: {torch.cuda.is_available()}')

## Step 3 — Build / Load Dataset

Replace the cell body with your own data-loading code if you have a real dataset.
The synthetic phrases below are intentionally diverse so the model is forced to
learn beyond keyword matching.

In [ ]:
# ── Synthetic dataset (replace with your real data) ─────────────────────────
INTENTS = {
    'CMD_ACADEMY': [
        'What courses are available?',
        'Show me the course catalogue',
        'I want to enrol in a training programme',
        'List all academy subjects',
        'Register me for the leadership workshop',
        'How do I sign up for online learning?',
        'Are there any upcoming webinars?',
        'I need certification training',
        'Find me a Python programming course',
        'Can I audit a class without enrolling?',
    ],
    'CMD_ACTIVITY': [
        'What activities are planned this week?',
        'Show me the events calendar',
        'I want to join the community event',
        'Are there any sports activities?',
        'Register me for the team-building exercise',
        'List upcoming workshops',
        'What is happening this weekend?',
        'I am interested in volunteering opportunities',
        'Schedule a group activity for my team',
        'How do I participate in the annual retreat?',
    ],
    'CMD_ADS': [
        'I want to advertise my business',
        'How do I post an advertisement?',
        'Place an ad for my product',
        'Show me advertising options',
        'What are the rates for a banner ad?',
        'Promote my service on this platform',
        'I need to run a marketing campaign',
        'Create a sponsored listing for my company',
        'How do I boost my post visibility?',
        'Submit an advertisement request',
    ],
    'CMD_BOOKING': [
        'Book an appointment for next Monday',
        'I need to reserve a meeting room',
        'Schedule a consultation with the doctor',
        'Can I make a booking for tomorrow afternoon?',
        'Reserve a slot in the conference hall',
        'I want to set up an appointment',
        'Book a table for four people',
        'Schedule a follow-up session',
        'Arrange a meeting with the manager',
        'I need to book a facility for Friday',
    ],
    'CMD_DISCOVERY': [
        'Show me available facilities',
        'What services are offered here?',
        'Help me find the nearest clinic',
        'Explore what is available in my area',
        'I am looking for local resources',
        'Discover community programmes',
        'What can I do in this app?',
        'Find me nearby support centres',
        'Browse available options',
        'List all features I can use',
    ],
    'CMD_FACILITY_MGMT': [
        'Report a maintenance issue in block B',
        'The air conditioning is broken',
        'Request a cleaning service for the office',
        'I need a technician for the conference room',
        'Submit a facility repair request',
        'The projector in hall 3 is not working',
        'Schedule preventive maintenance',
        'Who manages building facilities?',
        'Log a complaint about the elevator',
        'Arrange an inspection of the premises',
    ],
    'CMD_MEDICAL': [
        'I need medical advice',
        'What are the symptoms of diabetes?',
        'Connect me with a doctor',
        'I have a health question',
        'Show me nearby hospitals',
        'I need an emergency contact number',
        'Provide information about medication',
        'I feel unwell, what should I do?',
        'Find a specialist for back pain',
        'What vaccinations are recommended for travel?',
    ],
    'CMD_ORG': [
        'Who is in charge of this organisation?',
        'Show me the organisational chart',
        'List all departments',
        'Who is my direct manager?',
        'Find contact details for HR',
        'What are the working hours?',
        'Introduce me to the leadership team',
        'Who should I report to?',
        'Company policies and guidelines',
        'Show me the staff directory',
    ],
    'INTENT_ASK_AI': [
        'Can you help me with a question?',
        'Tell me a joke',
        'What is the capital of France?',
        'Explain machine learning to me',
        'Write a short poem',
        'Summarise this document for me',
        'Translate this sentence to Arabic',
        'Help me draft an email',
        'What is the weather like today?',
        'Give me a fun fact',
    ],
    'UNKNOWN_INTENT': [
        'asdfghjkl',
        '12345 random text',
        'blah blah blah',
        'xyzzy plugh',
        'I do not know what I want',
        'just testing',
        'nothing in particular',
        'gibberish input here',
        '???',
        'hello world',
    ],
}

# Expand dataset by repeating samples (simulates larger synthetic corpus)
REPEAT = 500   # adjust to your available memory
rows = []
for label, phrases in INTENTS.items():
    for _ in range(REPEAT):
        for phrase in phrases:
            rows.append({'text': phrase, 'label': label})

combined_df = pd.DataFrame(rows).sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f'Total samples : {len(combined_df)}')
print(combined_df['label'].value_counts())

## Step 4 — Encode Labels & Split

**Key fixes:**
- Label mapping is saved to disk so inference always uses the correct label order.
- 3-way stratified split: **80 % train / 10 % val / 10 % test**.

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from datasets import Dataset
import json

le = LabelEncoder()
combined_df['label_int'] = le.fit_transform(combined_df['label'])

label_names = list(le.classes_)           # alphabetically sorted
id2label    = {i: l for i, l in enumerate(label_names)}
label2id    = {l: i for i, l in enumerate(label_names)}

print('Label mapping:')
for k, v in id2label.items():
    print(f'  {k}  →  {v}')

# ── Save label mapping to disk ───────────────────────────────────────────────
# FIX: Without this, inference may silently use a different label order.
with open('/content/label_mapping.json', 'w') as f:
    json.dump(
        {'id2label': id2label, 'label2id': label2id, 'label_names': label_names},
        f,
        indent=2,
    )
print('\n✅ Label mapping saved to /content/label_mapping.json')

# ── 3-way stratified split: 80 / 10 / 10 ───────────────────────────────────
work = combined_df[['text', 'label_int']].rename(columns={'label_int': 'label'})

train_df, temp_df = train_test_split(
    work, test_size=0.2, random_state=SEED, stratify=work['label']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=SEED, stratify=temp_df['label']
)

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
val_ds   = Dataset.from_pandas(val_df.reset_index(drop=True))
# test_ds is kept as a pandas DataFrame for Step 10

print(f'\n📦 Train : {len(train_df)}')
print(f'📦 Val   : {len(val_df)}')
print(f'📦 Test  : {len(test_df)}')

## Step 5 — Tokenize

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length',
        max_length=MAX_LENGTH,
    )

train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize, batched=True)

train_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
val_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

print('✅ Tokenization complete')

## Step 6 — Fine-Tune

**Key fixes vs original:**
- `num_train_epochs=10` (down from 30 — model converges at epoch 1 on synthetic data)
- `warmup_ratio=0.05` instead of fixed `warmup_steps`
- `learning_rate=2e-5` (slightly lower for stability)
- `label_smoothing_factor=0.1` — prevents overconfident predictions that generalise poorly
- `early_stopping_patience=3` (down from 5)

In [ ]:
import evaluate as ev

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

acc_metric = ev.load('accuracy')
f1_metric  = ev.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': acc_metric.compute(predictions=preds, references=labels)['accuracy'],
        'f1':       f1_metric.compute(
                        predictions=preds, references=labels, average='weighted'
                    )['f1'],
    }

args = TrainingArguments(
    output_dir='./intent_model',
    num_train_epochs=10,                 # reduced from 30
    per_device_train_batch_size=16,   # adjust to your GPU memory (512 requires A100)
    per_device_eval_batch_size=64,    # adjust to your GPU memory
    warmup_ratio=0.05,                   # ratio-based warmup instead of fixed steps
    weight_decay=0.01,
    learning_rate=2e-5,                  # slightly lower LR
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_steps=50,
    fp16=True,
    report_to='none',
    label_smoothing_factor=0.1,          # prevents overconfident predictions
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()

## Step 7 — Evaluate on Validation Set

In [ ]:
val_results = trainer.evaluate(val_ds)
print('Validation results:')
for k, v in val_results.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

## Step 8 — Save Model + Tokenizer + Label Mapping

**New step** — saves everything needed for correct inference:
- model weights
- tokenizer (must match training tokenizer)
- `label_mapping.json` (prevents label-order bugs at inference time)

In [ ]:
SAVE_DIR = '/content/intent_classifier_final'
os.makedirs(SAVE_DIR, exist_ok=True)

# Save best model weights & tokenizer
trainer.model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

# Save label mapping alongside the model
with open(os.path.join(SAVE_DIR, 'label_mapping.json'), 'w') as f:
    json.dump({'id2label': id2label, 'label2id': label2id}, f, indent=2)

print(f'✅ Model, tokenizer, and label mapping saved to {SAVE_DIR}')

## Step 9 — Correct Inference Pipeline

**New step** — fixes all common inference mistakes:
- Loads `label_mapping.json` from disk (not hard-coded)
- Uses `int(k)` when rebuilding the id→label dict (JSON keys are always strings)
- Applies `torch.softmax` before `argmax` (raw logits ≠ probabilities)
- Uses identical `MAX_LENGTH`, `padding='max_length'` as training

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

SAVE_DIR   = '/content/intent_classifier_final'
MAX_LENGTH = 64  # MUST match training

# Load
infer_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
infer_model     = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR)
infer_model.eval()

# FIX: JSON keys are always strings; cast back to int so id2label lookup works
with open(os.path.join(SAVE_DIR, 'label_mapping.json')) as f:
    mapping = json.load(f)
infer_id2label = {int(k): v for k, v in mapping['id2label'].items()}


def predict_intent(texts):
    """
    Predict intent for one or more text strings.

    Parameters
    ----------
    texts : str or list[str]

    Returns
    -------
    list[dict]  –  each dict has 'label' and 'confidence'.
    """
    if isinstance(texts, str):
        texts = [texts]

    # FIX: tokenise exactly as during training
    enc = infer_tokenizer(
        texts,
        truncation=True,
        padding='max_length',
        max_length=MAX_LENGTH,
        return_tensors='pt',
    )

    with torch.no_grad():
        logits = infer_model(**enc).logits          # shape: (batch, num_labels)

    # FIX: apply softmax — raw logits are NOT probabilities
    probs       = torch.softmax(logits, dim=-1)
    pred_ids    = probs.argmax(dim=-1).tolist()
    confidences = probs.max(dim=-1).values.tolist()

    return [
        {'label': infer_id2label[pid], 'confidence': round(conf, 4)}
        for pid, conf in zip(pred_ids, confidences)
    ]


# ── Smoke test ───────────────────────────────────────────────────────────────
test_phrases = [
    'Book an appointment for next Monday',
    'I need medical advice',
    'Show me available facilities',
    'Who is in charge of this organisation?',
    'asdfghjkl random nonsense',
]
print('Smoke test results:')
for phrase in test_phrases:
    result = predict_intent(phrase)
    print(f"  '{phrase}'")
    print(f"   → {result[0]}")

## Step 10 — Evaluate on Held-Out Test Set

**New step** — final evaluation on data the model has never seen.

In [ ]:
from sklearn.metrics import classification_report
from datasets import Dataset
import numpy as np

test_ds_eval = Dataset.from_pandas(test_df.reset_index(drop=True))
test_ds_eval = test_ds_eval.map(tokenize, batched=True)
test_ds_eval.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

preds_output = trainer.predict(test_ds_eval)
preds        = np.argmax(preds_output.predictions, axis=-1)
true_labels  = preds_output.label_ids

print('📋 Test Set Classification Report:')
print(
    classification_report(
        true_labels,
        preds,
        target_names=label_names,
    )
)

## Step 11 — Export to TFLite

Loads from `SAVE_DIR` (the correctly saved model from Step 8).

In [ ]:
!pip install -q onnx onnxruntime tf2onnx tensorflow

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

SAVE_DIR   = '/content/intent_classifier_final'
ONNX_PATH  = '/content/intent_classifier.onnx'
MAX_LENGTH = 64

export_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
export_model     = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR)
export_model.eval()

dummy_input = export_tokenizer(
    'dummy sentence',
    return_tensors='pt',
    truncation=True,
    padding='max_length',
    max_length=MAX_LENGTH,
)

torch.onnx.export(
    export_model,
    (dummy_input['input_ids'], dummy_input['attention_mask']),
    ONNX_PATH,
    input_names=['input_ids', 'attention_mask'],
    output_names=['logits'],
    dynamic_axes={
        'input_ids':      {0: 'batch_size'},
        'attention_mask': {0: 'batch_size'},
        'logits':         {0: 'batch_size'},
    },
    opset_version=13,
)
print(f'✅ ONNX model exported to {ONNX_PATH}')

In [ ]:
import subprocess, os

TF_SAVED_MODEL = '/content/intent_classifier_tf'
TFLITE_PATH    = '/content/intent_classifier.tflite'

# Convert ONNX → TensorFlow SavedModel
subprocess.run(
    [
        'python', '-m', 'tf2onnx.convert',
        '--onnx', ONNX_PATH,
        '--output', TF_SAVED_MODEL,
        '--saved-model',
    ],
    check=True,
)

# Convert TF SavedModel → TFLite
import tensorflow as tf

converter = tf.lite.TFLiteConverter.from_saved_model(TF_SAVED_MODEL)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_model)

size_mb = os.path.getsize(TFLITE_PATH) / 1024 / 1024
print(f'✅ TFLite model saved to {TFLITE_PATH}  ({size_mb:.1f} MB)')

In [ ]:
import numpy as np
import tensorflow as tf

TFLITE_PATH = '/content/intent_classifier.tflite'

interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print('TFLite model inputs :')
for d in input_details:
    print(f"  {d['name']}  shape={d['shape']}  dtype={d['dtype']}")
print('TFLite model outputs:')
for d in output_details:
    print(f"  {d['name']}  shape={d['shape']}  dtype={d['dtype']}")

# Quick inference with TFLite
from transformers import AutoTokenizer
import json

tflite_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
with open(os.path.join(SAVE_DIR, 'label_mapping.json')) as f:
    tflite_mapping = json.load(f)
tflite_id2label = {int(k): v for k, v in tflite_mapping['id2label'].items()}

sample_text = 'Book an appointment'
enc = tflite_tokenizer(
    sample_text,
    truncation=True,
    padding='max_length',
    max_length=MAX_LENGTH,
    return_tensors='np',
)

interpreter.set_tensor(input_details[0]['index'], enc['input_ids'].astype(np.int32))
interpreter.set_tensor(input_details[1]['index'], enc['attention_mask'].astype(np.int32))
interpreter.invoke()

logits = interpreter.get_tensor(output_details[0]['index'])   # (1, num_labels)
probs  = np.exp(logits) / np.exp(logits).sum(axis=-1, keepdims=True)  # softmax
pred   = int(np.argmax(probs, axis=-1)[0])

print(f"\nTFLite prediction for '{sample_text}':")
print(f"  Label      : {tflite_id2label[pred]}")
print(f"  Confidence : {probs[0][pred]:.4f}")